<a href="https://colab.research.google.com/github/amitranjan74/Performance_Automation/blob/main/Extract_Table_Using_OCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project on extracting table Using OCR

In [ ]:
# Install system packages for OCR
!sudo apt-get update
!sudo apt-get install -y poppler-utils tesseract-ocr

# Install Python libraries for OCR
!pip install pdf2image pytesseract

import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import os

# Although deepseek API doesn't directly offer PDF table extraction, we can leverage LLMs (e.g. `google.colab.ai`) after extracting text using OCR.
from google.colab import ai
import io

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (2,895 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1

In [ ]:
# Define the path to your PDF file
pdf_path = "/content/Statement_CM.pdf"

# Check if the PDF file exists
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"The PDF file was not found at: {pdf_path}")

# Convert PDF to a list of images
print(f"Converting PDF '{pdf_path}' to images...")
images = convert_from_path(pdf_path)
print(f"Converted {len(images)} pages.")

ocr_text = ""
# Loop through each image and apply OCR
print("Extracting text from images using OCR...")
for i, image in enumerate(images):
    page_text = pytesseract.image_to_string(image)
    ocr_text += page_text + "\n" # Add newline between pages
    print(f"  - Page {i+1} OCR complete.")

print("\nOCR Extracted Text:")
print(ocr_text[:1000]) # Print first 1000 characters for brevity

Converting PDF '/content/Statement_CM.pdf' to images...
Converted 8 pages.
Extracting text from images using OCR...
  - Page 1 OCR complete.
  - Page 2 OCR complete.
  - Page 3 OCR complete.
  - Page 4 OCR complete.
  - Page 5 OCR complete.
  - Page 6 OCR complete.
  - Page 7 OCR complete.
  - Page 8 OCR complete.

OCR Extracted Text:
(A Depository Participant of NSDL)DP ID IN301549

Holding Statement as on : 17/03/2026 Business Date: 18/03/2026

DP Account No : 14726073
THIAGARAJAN ARAVAMUDAN

SUBAGHA

NO 131 (NEW 55) GANAPATHY NAGAR
T V KOIL POST

TRICHY

620005

Account Type : Free Balance

Sr. No. ISIN Company Name Scrip Balance Market Market Value
Type Rate

EQ
31 INFOTECH
INE748C01038 LIMITED FV RS. 65,500.000 13.150 8,61,325.000
10/-

| 2 1INE470A01017 |3M INDIA LIMITED 110.000] 33,148.100] 36,46,291.000

EQ
ABB INDIA
INE117A01022 LIMITED 800.000} 6,398.800] 51,19,040.000
ADITYA BIRLA
INE674K01013 CAPITAL LIMITED 900.000 324.750 2,92,275.000

ADITYA BIRLA
5 | INE647001011 | FASH

Now that we have the text extracted using OCR, we'll pass it to `google.colab.ai` (accessing the Deepseek model) to identify and extract any tabular data into a JSON format.

In [ ]:
prompt_ocr = f"""
From the following text, extract any tabular data and present it in a structured JSON format.
If there are multiple tables, list them as separate objects in a JSON array.
If no table is found, return an empty JSON object.

Text:
{ocr_text}
"""

try:
    response_ocr = ai.generate_text(prompt=prompt_ocr)
    print("\nLLM Response (Extracted Table from OCR text):")
    print(response_ocr)
except Exception as e:
    print(f"\nAn error occurred while calling the AI model with OCR text: {e}")
    print("Please ensure you have access to the google.colab.ai service and your prompt is well-formed.")

print("\n---")


LLM Response (Extracted Table from OCR text):
```json
[
  {
    "Account Type": "Free Balance",
    "data": [
      {
        "Sr. No.": null,
        "ISIN": "INE748C01038",
        "Company Name": "31 INFOTECH LIMITED",
        "Scrip Type": "EQ FV RS. 10/-",
        "Balance": 65500.0,
        "Market Rate": 13.15,
        "Market Value": 861325.0,
        "Status": "Free"
      },
      {
        "Sr. No.": 2,
        "ISIN": "INE470A01017",
        "Company Name": "3M INDIA LIMITED",
        "Scrip Type": null,
        "Balance": 110.0,
        "Market Rate": 33148.1,
        "Market Value": 3646291.0,
        "Status": "Free"
      },
      {
        "Sr. No.": null,
        "ISIN": "INE117A01022",
        "Company Name": "ABB INDIA LIMITED",
        "Scrip Type": "EQ",
        "Balance": 800.0,
        "Market Rate": 6398.8,
        "Market Value": 5119040.0,
        "Status": "Free"
      },
      {
        "Sr. No.": null,
        "ISIN": "INE674K01013",
        "Company Name

In [ ]:
import pandas as pd
import json

# Assuming response_ocr contains the JSON string from the previous step
# It's good practice to ensure it's a string before parsing
json_data_str = response_ocr

# Some LLMs might wrap JSON in markdown code blocks, so we'll try to extract it.
if json_data_str.strip().startswith('```json') and json_data_str.strip().endswith('```'):
    json_data_str = json_data_str.strip()[len('```json'):-len('```')]

In [ ]:
data = json.loads(json_data_str)

In [ ]:
data

[{'Account Type': 'Free Balance',
  'data': [{'Sr. No.': None,
    'ISIN': 'INE748C01038',
    'Company Name': '31 INFOTECH LIMITED',
    'Scrip Type': 'EQ FV RS. 10/-',
    'Balance': 65500.0,
    'Market Rate': 13.15,
    'Market Value': 861325.0,
    'Status': 'Free'},
   {'Sr. No.': 2,
    'ISIN': 'INE470A01017',
    'Company Name': '3M INDIA LIMITED',
    'Scrip Type': None,
    'Balance': 110.0,
    'Market Rate': 33148.1,
    'Market Value': 3646291.0,
    'Status': 'Free'},
   {'Sr. No.': None,
    'ISIN': 'INE117A01022',
    'Company Name': 'ABB INDIA LIMITED',
    'Scrip Type': 'EQ',
    'Balance': 800.0,
    'Market Rate': 6398.8,
    'Market Value': 5119040.0,
    'Status': 'Free'},
   {'Sr. No.': None,
    'ISIN': 'INE674K01013',
    'Company Name': 'ADITYA BIRLA CAPITAL LIMITED',
    'Scrip Type': None,
    'Balance': 900.0,
    'Market Rate': 324.75,
    'Market Value': 292275.0,
    'Status': 'Free'},
   {'Sr. No.': 5,
    'ISIN': 'INE647001011',
    'Company Name': 'AD

In [ ]:


try:

    # Initialize an empty list to store DataFrames
    dataframes = []

    # Iterate through the list of tables (each table is an object in the list)
    for table_obj in data:
        # Check if 'data' key exists and is a list of dictionaries
        if 'data' in table_obj and isinstance(table_obj['data'], list):
            df = pd.DataFrame(table_obj['data'])
            # Add a title if available
            if 'title' in table_obj:
                df.name = table_obj['title']
            elif 'Account Type' in table_obj:
                df.name = f"Table for Account Type: {table_obj['Account Type']}"
            dataframes.append(df)
        else:
            print(f"Skipping object as 'data' key is missing or not a list: {table_obj}")

    # Display each created DataFrame
    if dataframes:
        for i, df in enumerate(dataframes):
            print(f"\n--- Table {i+1}: {getattr(df, 'name', 'Untitled Table')} ---")
            df.to_csv(f"table_{i+1}.csv", index=False)
            display(df.head())
    else:
        print("No dataframes could be created from the JSON.")

except json.JSONDecodeError as e:
    print(f"Error decoding JSON: {e}")
    print("Please ensure the LLM response is valid JSON.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


--- Table 1: Table for Account Type: Free Balance ---


,Sr. No.,ISIN,Company Name,Scrip Type,Balance,Market Rate,Market Value,Status
0,NaN,INE748C01038,31 INFOTECH LIMITED,EQ FV RS. 10/-,65500.0,13.15,861325.0,Free
1,2.0,INE470A01017,3M INDIA LIMITED,None,110.0,33148.10,3646291.0,Free
2,NaN,INE117A01022,ABB INDIA LIMITED,EQ,800.0,6398.80,5119040.0,Free
3,NaN,INE674K01013,ADITYA BIRLA CAPITAL LIMITED,None,900.0,324.75,292275.0,Free
4,5.0,INE647001011,ADITYA BIRLA FASHION AND RETAIL LIMITED,EQ,2000.0,59.04,118080.0,Free



--- Table 2: Table for Account Type: Pledge a/c ---


,Sr. No.,ISIN,Company Name,Scrip Type,Balance,Market Rate,Market Value,Status
0,NaN,INE029A01011,BHARAT PETROLEUM CORPORATION LIMITED,EQ,15000.0,303.30,4549500.0,None
1,2.0,INE081A01020,TATA STEEL LIMITED,EQ NEW 5.7 pp 1).,33300.0,195.05,6495165.0,None
